# Voidmetric 3D Anime What-If — Google Colab Pro GPU Renderer (`L4` / `T4`)

Render your Remotion + Three.js 3D Anime Short at **full native 2K (`1440x2560`, 30 fps)** or **1080p (`1080x1920`, 30 fps)** in **~2–4 minutes** using Google Colab Pro's hardware GPU!

### Recommended Runtime Setup
1. **Runtime → Change runtime type → Hardware accelerator:**
   - 🏆 **Best:** `L4 GPU` (24 GB VRAM, hardware 3D rasterization, fast & economical)
   - 💰 **Budget:** `T4 GPU` (16 GB VRAM, lowest compute unit burn)
   - ❌ **Avoid:** `A100` (compute-only, wastes units) and `TPUs` (no WebGL support)
2. Run Cells 1 → 5 in order (or **Runtime → Run all** after filling Cell 2).

In [ ]:
#@title 1. Verify NVIDIA GPU & Install Node 20 + Headless Chromium EGL Libraries
import os, shutil, subprocess, sys

gpu_check = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if gpu_check.returncode == 0:
    print('GPU DETECTED:', gpu_check.stdout.strip())
    if 'A100' in gpu_check.stdout:
        print('TIP: L4 or T4 GPUs are better & cheaper than A100 for Three.js WebGL rendering!')
else:
    print('WARNING: No NVIDIA GPU detected! Switch Runtime -> Change runtime type -> L4 GPU or T4 GPU for 10x faster rendering.')

# Ensure Node 20+ is installed (Colab default can be older)
node_v = subprocess.run(['node', '-v'], capture_output=True, text=True)
need_node = node_v.returncode != 0 or int(node_v.stdout.strip().lstrip('v').split('.')[0]) < 18
if need_node:
    !curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - >/dev/null
    !sudo apt-get install -y -qq nodejs >/dev/null
print('Node:', subprocess.check_output(['node', '-v'], text=True).strip(), '| npm:', subprocess.check_output(['npm', '-v'], text=True).strip())

# Headless Chromium + EGL/GLES libraries for Remotion hardware WebGL (--gl=angle-egl)
!sudo apt-get update -qq
!sudo apt-get install -y -qq ffmpeg libgbm1 libnss3 libatk-bridge2.0-0 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libpango-1.0-0 libcairo2 libegl1 libgles2 >/dev/null 2>&1 || true
!pip install -q numpy scipy pillow
print('System environment ready!')

In [ ]:
#@title 2. Configure Episode Source & Render Settings
EPISODE_SOURCE = "upload_zip" #@param ["upload_zip", "google_drive_zip", "github"]
GITHUB_REPO_URL = "https://github.com/YOUR-USERNAME/anime-whatif-3d-voidmetric.git" #@param {type:"string"}
GITHUB_BRANCH = "main" #@param {type:"string"}
REMOTION_SUBDIR = "workspace-remotion" #@param {type:"string"}
DRIVE_ZIP_PATH = "/content/drive/MyDrive/Voidmetric_Uploads/episode_workspace.zip" #@param {type:"string"}

RESOLUTION_PROFILE = "1080p (1080x1920)" #@param ["1080p (1080x1920)", "2K (1440x2560)"]
COMPOSITION_ID = "EpisodeShort1080" #@param {type:"string"}
EPISODE_SLUG = "gojo-limitless" #@param {type:"string"}
CRF_QUALITY = 16 #@param {type:"slider", min:14, max:22, step:1}
CONCURRENCY = 4 #@param {type:"slider", min:1, max:6, step:1}
SET_GPU_MSAA_4 = True #@param {type:"boolean"}
SAVE_TO_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/Voidmetric_Renders" #@param {type:"string"}

import os, zipfile, glob, shutil, pathlib

WORK_ROOT = pathlib.Path('/content/voidmetric_work')
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

if SAVE_TO_GOOGLE_DRIVE or EPISODE_SOURCE == 'google_drive_zip':
    from google.colab import drive
    drive.mount('/content/drive')

if EPISODE_SOURCE == 'github':
    !git clone --depth 1 -b {GITHUB_BRANCH} {GITHUB_REPO_URL} {WORK_ROOT}/repo
    REMOTION_DIR = WORK_ROOT / 'repo' / REMOTION_SUBDIR if REMOTION_SUBDIR else WORK_ROOT / 'repo'
elif EPISODE_SOURCE == 'google_drive_zip':
    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zf:
        zf.extractall(WORK_ROOT / 'unzipped')
    pkgs = list((WORK_ROOT / 'unzipped').rglob('package.json'))
    assert pkgs, 'Could not find package.json inside zip!'
    REMOTION_DIR = pkgs[0].parent
else:
    from google.colab import files
    print('Upload the episode workspace .zip exported from Gemini Tasks (containing package.json, src/, projects/)...')
    uploaded = files.upload()
    zip_name = next(iter(uploaded.keys()))
    with zipfile.ZipFile(zip_name, 'r') as zf:
        zf.extractall(WORK_ROOT / 'unzipped')
    pkgs = list((WORK_ROOT / 'unzipped').rglob('package.json'))
    assert pkgs, 'Could not find package.json inside uploaded zip!'
    REMOTION_DIR = pkgs[0].parent

# Optionally upgrade EFFECT_MSAA from 0 (cloud CPU) -> 4 (Colab L4/T4 GPU) for maximum sharpness!
if SET_GPU_MSAA_4:
    for tsx in REMOTION_DIR.rglob('EpisodeShort.tsx'):
        txt = tsx.read_text(encoding='utf-8')
        if 'const EFFECT_MSAA = 0;' in txt:
            tsx.write_text(txt.replace('const EFFECT_MSAA = 0;', 'const EFFECT_MSAA = 4;'), encoding='utf-8')
            print('Upgraded MSAA 0 -> 4 for GPU rendering in:', tsx)

print('REMOTION_DIR:', REMOTION_DIR)

In [ ]:
#@title 3. Install Node Packages & Auto-Detect Fastest Working GPU WebGL Mode
import os, subprocess, pathlib

TMP_DIR = pathlib.Path('/tmp/remotion-tmp')
TMP_DIR.mkdir(parents=True, exist_ok=True)
os.environ['REMOTION_TMP'] = str(TMP_DIR)
os.environ['TEMP'] = str(TMP_DIR)
os.environ['TMP'] = str(TMP_DIR)

if not (REMOTION_DIR / 'node_modules').exists():
    print('Installing npm packages (first run only, ~45s)...')
    subprocess.run(['npm', 'install', '--no-audit', '--no-fund'], cwd=str(REMOTION_DIR), check=True)

# Probe fastest hardware WebGL backend on Colab NVIDIA L4/T4, with clean fallback
SELECTED_GL = None
for candidate in ['angle-egl', 'egl', 'swangle']:
    env = os.environ.copy()
    env['REMOTION_GL'] = candidate
    print(f'Testing 1-frame render with --gl={candidate} ...')
    p = subprocess.run([
        'npx', 'remotion', 'still', 'src/index.ts', COMPOSITION_ID, '/tmp/gpu_probe.jpg',
        '--frame=0', '--image-format=jpeg', f'--gl={candidate}', '--log=error'
    ], cwd=str(REMOTION_DIR), env=env, capture_output=True, text=True)
    if p.returncode == 0 and os.path.exists('/tmp/gpu_probe.jpg') and os.path.getsize('/tmp/gpu_probe.jpg') > 5000:
        SELECTED_GL = candidate
        print(f'SELECTED_GL = {SELECTED_GL} (1-frame GPU smoke test PASSED!)')
        break
    else:
        print(f'  {candidate} failed:', (p.stderr or p.stdout).strip()[:240])

assert SELECTED_GL is not None, 'All WebGL render backends failed! Check src/Root.tsx and composition ID.'
from IPython.display import Image, display
display(Image('/tmp/gpu_probe.jpg', width=280))

In [ ]:
#@title 4. Render Full Video on Colab GPU & Mux Mastered Audio
import glob, os, shutil, subprocess, time, pathlib

proj_dir = REMOTION_DIR / 'projects' / EPISODE_SLUG
out_dir = proj_dir / 'out'
out_dir.mkdir(parents=True, exist_ok=True)

# Auto-locate mastered mix wav (mix.wav / mix_v2.wav)
mix_candidates = sorted(glob.glob(str(proj_dir / 'audio' / 'mix*.wav')), key=os.path.getmtime, reverse=True)
AUDIO_WAV = mix_candidates[0] if mix_candidates else None
print('Mastered Audio:', AUDIO_WAV or 'NONE FOUND (will render video-only)')

tag = '2K' if '2K' in RESOLUTION_PROFILE else '1080p'
SILENT_MP4 = out_dir / f'{EPISODE_SLUG}_silent_{tag}.mp4'
FINAL_MP4 = out_dir / f'{EPISODE_SLUG}-Voidmetric-3D-{tag}.mp4'

env = os.environ.copy()
env['REMOTION_GL'] = SELECTED_GL
eff_conc = CONCURRENCY if SELECTED_GL != 'swangle' else 1

t0 = time.time()
print(f'Rendering {COMPOSITION_ID} with --gl={SELECTED_GL} --concurrency={eff_conc} --crf={CRF_QUALITY} ...')
subprocess.run([
    'npx', 'remotion', 'render', 'src/index.ts', COMPOSITION_ID, str(SILENT_MP4),
    '--codec=h264', f'--crf={CRF_QUALITY}', '--pixel-format=yuv420p',
    f'--gl={SELECTED_GL}', f'--concurrency={eff_conc}', '--log=info'
], cwd=str(REMOTION_DIR), env=env, check=True)

if AUDIO_WAV:
    subprocess.run([
        'ffmpeg', '-y', '-hide_banner', '-loglevel', 'error',
        '-i', str(SILENT_MP4), '-i', AUDIO_WAV,
        '-map', '0:v:0', '-map', '1:a:0',
        '-c:v', 'copy', '-c:a', 'aac', '-b:a', '256k',
        '-movflags', '+faststart', '-shortest', str(FINAL_MP4)
    ], check=True)
    SILENT_MP4.unlink(missing_ok=True)
else:
    shutil.move(str(SILENT_MP4), str(FINAL_MP4))

# Clean Remotion temp bundles
for pat in ['remotion-webpack-bundle-*', 'puppeteer_dev_chrome_profile-*']:
    for m in glob.glob(str(TMP_DIR / pat)):
        shutil.rmtree(m, ignore_errors=True)

elapsed = time.time() - t0
size_mb = FINAL_MP4.stat().st_size / (1024 * 1024)
print(f'DONE in {elapsed/60:.2f} min! Final MP4: {FINAL_MP4} ({size_mb:.2f} MB)')

In [ ]:
#@title 5. Run QA Audit, Preview Contact Sheet & Export to Google Drive / Download
import glob, os, shutil, subprocess, pathlib
from IPython.display import Image, display

# Run short_qa.py if present in repo or workspace
qa_scripts = list(WORK_ROOT.rglob('short_qa.py'))
shots_files = sorted(glob.glob(str(REMOTION_DIR / 'projects' / EPISODE_SLUG / 'out' / 'shots*.json')))
qa_dir = REMOTION_DIR / 'projects' / EPISODE_SLUG / 'out' / 'qa_colab'

if qa_scripts:
    cmd = ['python3', str(qa_scripts[0]), str(FINAL_MP4), str(qa_dir)]
    if shots_files:
        cmd.append(shots_files[-1])
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout)
    if res.stderr:
        print(res.stderr)
    sheets = sorted(glob.glob(str(qa_dir / 'sheet_*.jpg')))
    if sheets:
        print('Sample QA sheet:', sheets[len(sheets)//2])
        display(Image(sheets[len(sheets)//2], width=760))

if SAVE_TO_GOOGLE_DRIVE:
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    dst = os.path.join(DRIVE_OUTPUT_DIR, FINAL_MP4.name)
    shutil.copy2(str(FINAL_MP4), dst)
    print('Saved final video to Google Drive:', dst)
else:
    from google.colab import files
    files.download(str(FINAL_MP4))